# Advanced A/B Testing — Sequential Testing & Multi-Armed Bandits

Classical A/B testing has blind spots: peeking at results early inflates false positive rates, and fixed sample sizes waste traffic on bad variants. This notebook covers modern solutions.

**Topics:** Sequential testing (SPRT), always-valid p-values, Thompson sampling, UCB, Bayesian A/B testing, multi-armed bandits

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

# True conversion rates (unknown in real experiments)
p_control  = 0.10  # 10% control conversion rate
p_treatment = 0.12  # 12% treatment (20% relative lift)

print('A/B Test Setup:')
print(f'  Control   conversion rate: {p_control:.1%}')
print(f'  Treatment conversion rate: {p_treatment:.1%}')
print(f'  True lift: {(p_treatment-p_control)/p_control:.1%} (relative)')
print(f'  MDE:       {p_treatment - p_control:.2%} (absolute)')

## 1. The Peeking Problem — Why Classical A/B Tests Break

In [ ]:
# Simulation: experimenters who peek daily and stop if p < 0.05
# Under null hypothesis (p_control == p_treatment), how often do they falsely conclude significance?

def simulate_peeking(n_simulations=5000, n_days=30, daily_users=100, alpha=0.05):
    """False positive rate when peeking daily vs waiting for pre-planned n."""
    fp_peeking  = 0  # false positives (peeking)
    fp_fixed    = 0  # false positives (fixed sample)
    
    # Under null: both have p_control conversion rate
    p_null = 0.10
    
    for _ in range(n_simulations):
        ctrl_conv, treat_conv = 0, 0
        ctrl_n, treat_n = 0, 0
        stopped_early = False
        
        for day in range(n_days):
            # New arrivals each day
            ctrl_new   = np.random.binomial(daily_users, p_null)
            treat_new  = np.random.binomial(daily_users, p_null)
            ctrl_conv  += ctrl_new
            treat_conv += treat_new
            ctrl_n     += daily_users
            treat_n    += daily_users
            
            # Peek: run z-test
            if ctrl_n > 0 and treat_n > 0 and not stopped_early:
                p_c = ctrl_conv / ctrl_n
                p_t = treat_conv / treat_n
                se  = np.sqrt(p_null*(1-p_null)*(1/ctrl_n + 1/treat_n))
                if se > 0:
                    z   = (p_t - p_c) / se
                    p_val = 2 * (1 - stats.norm.cdf(abs(z)))
                    if p_val < alpha:
                        fp_peeking += 1
                        stopped_early = True
        
        # Fixed sample: only look at end
        p_c_final = ctrl_conv / ctrl_n
        p_t_final = treat_conv / treat_n
        se_final  = np.sqrt(p_null*(1-p_null)*(2/ctrl_n))
        if se_final > 0:
            z_final = (p_t_final - p_c_final) / se_final
            p_final = 2 * (1 - stats.norm.cdf(abs(z_final)))
            if p_final < alpha:
                fp_fixed += 1
    
    return fp_peeking / n_simulations, fp_fixed / n_simulations

fpr_peek, fpr_fixed = simulate_peeking(n_simulations=3000)

print('=== Peeking Problem Demonstration ===')
print(f'  Nominal α:         0.050 (5%)')
print(f'  Fixed sample FPR:  {fpr_fixed:.3f} ({fpr_fixed:.1%}) — correct!')
print(f'  Daily peeking FPR: {fpr_peek:.3f} ({fpr_peek:.1%}) — INFLATED!')
print(f'\n  Peeking inflates FPR by {fpr_peek/0.05:.1f}x vs nominal α!')
print('\n  Solutions:')
print('    1. Sequential testing (SPRT / always-valid p-values)')
print('    2. Bayesian A/B testing with posterior monitoring')
print('    3. Multi-armed bandits (adaptive allocation)')

## 2. Sequential Probability Ratio Test (SPRT)

In [ ]:
def sprt_test(conversions_c, n_c, conversions_t, n_t, p0, p1, alpha=0.05, beta=0.20):
    """
    Wald SPRT for proportions.
    H0: p = p0 (no effect)
    H1: p = p1 (minimum detectable effect)
    Returns: 'H1' (significant), 'H0' (no effect), or 'continue'
    """
    # Wald boundaries
    A = np.log((1 - beta) / alpha)   # upper boundary (reject H0)
    B = np.log(beta / (1 - alpha))   # lower boundary (accept H0)
    
    # Log-likelihood ratio
    if n_c == 0 or n_t == 0:
        return 'continue', 0.0
    
    p_c = conversions_c / n_c
    p_t = conversions_t / n_t
    
    # Clamp to avoid log(0)
    p_c = np.clip(p_c, 1e-9, 1-1e-9)
    p_t = np.clip(p_t, 1e-9, 1-1e-9)
    
    llr = (conversions_t * np.log(p1/p0) +
           (n_t - conversions_t) * np.log((1-p1)/(1-p0)))
    
    if llr >= A:
        return 'H1', llr
    elif llr <= B:
        return 'H0', llr
    else:
        return 'continue', llr

# Simulate a true A/B test with sequential monitoring
daily_users = 200  # total per day
n_days = 40
p0 = p_control
p1 = p_treatment

llr_history = []
n_total_c, n_total_t = 0, 0
conv_c, conv_t = 0, 0
decision = 'continue'
decision_day = None

A = np.log(0.8 / 0.05)  # boundaries for α=0.05, β=0.20
B = np.log(0.20 / 0.95)

for day in range(1, n_days + 1):
    n_c = daily_users // 2
    n_t = daily_users // 2
    conv_c += np.random.binomial(n_c, p_control)
    conv_t += np.random.binomial(n_t, p_treatment)
    n_total_c += n_c
    n_total_t += n_t
    
    result, llr = sprt_test(conv_c, n_total_c, conv_t, n_total_t, p0, p1)
    llr_history.append(llr)
    
    if result != 'continue' and decision == 'continue':
        decision = result
        decision_day = day

print(f'SPRT Decision: {decision} on day {decision_day}')
print(f'Total users used: {n_total_c + n_total_t}')
print(f'True answer: H1 (treatment is better)')

plt.figure(figsize=(12, 5))
days = range(1, len(llr_history)+1)
plt.plot(days, llr_history, 'steelblue', lw=2, label='Log Likelihood Ratio')
plt.axhline(A, color='red', ls='--', lw=2, label=f'H1 boundary (+{A:.2f})')
plt.axhline(B, color='orange', ls='--', lw=2, label=f'H0 boundary ({B:.2f})')
plt.axhline(0, color='gray', ls=':', alpha=0.5)
if decision_day:
    plt.axvline(decision_day, color='green', ls=':', lw=2, label=f'Decision: Day {decision_day}')
plt.xlabel('Day'); plt.ylabel('Log Likelihood Ratio')
plt.title(f'SPRT Sequential Test — Decision: {decision} (Day {decision_day})')
plt.legend(); plt.tight_layout(); plt.show()

## 3. Bayesian A/B Testing

In [ ]:
# Bayesian approach: model posterior distribution over conversion rates
# Beta-Binomial conjugate: Beta(alpha, beta) prior + Binomial likelihood
# Posterior: Beta(alpha + conversions, beta + non-conversions)

from scipy.stats import beta as beta_dist

def bayesian_ab_test(conv_c, n_c, conv_t, n_t, prior_alpha=1, prior_beta=1, n_samples=100_000):
    """Returns P(treatment > control) using Monte Carlo sampling."""
    # Posterior parameters
    post_alpha_c = prior_alpha + conv_c
    post_beta_c  = prior_beta  + (n_c - conv_c)
    post_alpha_t = prior_alpha + conv_t
    post_beta_t  = prior_beta  + (n_t - conv_t)
    
    # Sample from posteriors
    samples_c = beta_dist.rvs(post_alpha_c, post_beta_c, size=n_samples)
    samples_t = beta_dist.rvs(post_alpha_t, post_beta_t, size=n_samples)
    
    prob_better = (samples_t > samples_c).mean()
    expected_lift = (samples_t - samples_c).mean()
    ci_lift = np.percentile(samples_t - samples_c, [2.5, 97.5])
    
    return prob_better, expected_lift, ci_lift, (post_alpha_c, post_beta_c), (post_alpha_t, post_beta_t)

# Simulate accumulation of data day by day
daily_n = 100  # per variant per day
prob_better_history = []
lift_history = []
conv_c_total, conv_t_total = 0, 0
n_c_total, n_t_total = 0, 0

for day in range(1, 31):
    conv_c_total += np.random.binomial(daily_n, p_control)
    conv_t_total += np.random.binomial(daily_n, p_treatment)
    n_c_total += daily_n
    n_t_total += daily_n
    
    prob, lift, ci, params_c, params_t = bayesian_ab_test(
        conv_c_total, n_c_total, conv_t_total, n_t_total
    )
    prob_better_history.append(prob)
    lift_history.append(lift * 100)  # pct

# Final results
prob, lift, ci, params_c, params_t = bayesian_ab_test(
    conv_c_total, n_c_total, conv_t_total, n_t_total
)

print(f'Control:   {conv_c_total}/{n_c_total} = {conv_c_total/n_c_total:.2%}')
print(f'Treatment: {conv_t_total}/{n_t_total} = {conv_t_total/n_t_total:.2%}')
print(f'P(treatment > control): {prob:.1%}')
print(f'Expected lift: {lift*100:+.2f}pp  [95% CI: {ci[0]*100:.2f}pp, {ci[1]*100:.2f}pp]')

# Visualize posteriors
x = np.linspace(0.04, 0.20, 1000)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pdf_c = beta_dist.pdf(x, *params_c)
pdf_t = beta_dist.pdf(x, *params_t)
axes[0].fill_between(x, pdf_c, alpha=0.4, color='steelblue', label=f'Control  μ={params_c[0]/(params_c[0]+params_c[1]):.3f}')
axes[0].fill_between(x, pdf_t, alpha=0.4, color='orange',    label=f'Treatment μ={params_t[0]/(params_t[0]+params_t[1]):.3f}')
axes[0].set_xlabel('Conversion Rate'); axes[0].set_ylabel('Posterior Density')
axes[0].set_title(f'Posterior Distributions\nP(T>C) = {prob:.1%}')
axes[0].legend()

# P(treatment better) over time
axes[1].plot(range(1, 31), [p*100 for p in prob_better_history], 'steelblue', lw=2)
axes[1].axhline(95, color='red', ls='--', label='95% threshold')
axes[1].axhline(50, color='gray', ls=':', alpha=0.5)
axes[1].set_xlabel('Day'); axes[1].set_ylabel('P(Treatment > Control) %')
axes[1].set_title('Bayesian Credibility Over Time'); axes[1].legend()
axes[1].set_ylim(0, 100)

plt.tight_layout(); plt.show()

## 4. Multi-Armed Bandits

In [ ]:
# Multi-armed bandits: explore vs exploit trade-off
# Instead of splitting 50/50, adaptively allocate more traffic to better variants

class Bandit:
    """Bernoulli bandit (simulates a website variant)."""
    def __init__(self, true_p, name):
        self.true_p = true_p
        self.name = name
        self.n_pulls = 0
        self.n_success = 0
    
    def pull(self):
        reward = int(np.random.random() < self.true_p)
        self.n_pulls += 1
        self.n_success += reward
        return reward
    
    @property
    def empirical_p(self):
        return self.n_success / max(self.n_pulls, 1)

# 4 variants (arms)
true_rates = [0.10, 0.12, 0.11, 0.15]  # variant D is best!
names = ['A (control)', 'B', 'C', 'D (best)']
n_rounds = 5000

# --- Strategy 1: Epsilon-Greedy ---
def epsilon_greedy(bandits, n_rounds, epsilon=0.1):
    bandits = [Bandit(p, n) for p, n in zip(true_rates, names)]
    rewards = []
    choices = []
    for _ in range(n_rounds):
        if np.random.random() < epsilon:
            arm = np.random.randint(len(bandits))
        else:
            arm = np.argmax([b.empirical_p for b in bandits])
        rewards.append(bandits[arm].pull())
        choices.append(arm)
    return rewards, choices, bandits

# --- Strategy 2: UCB1 ---
def ucb1(bandits_init, n_rounds):
    bandits = [Bandit(p, n) for p, n in zip(true_rates, names)]
    rewards = []
    choices = []
    # Pull each arm once first
    for i, b in enumerate(bandits):
        rewards.append(b.pull()); choices.append(i)
    for t in range(len(bandits), n_rounds):
        ucb = [b.empirical_p + np.sqrt(2 * np.log(t) / b.n_pulls) for b in bandits]
        arm = np.argmax(ucb)
        rewards.append(bandits[arm].pull()); choices.append(arm)
    return rewards, choices, bandits

# --- Strategy 3: Thompson Sampling (Bayesian) ---
def thompson_sampling(bandits_init, n_rounds):
    bandits = [Bandit(p, n) for p, n in zip(true_rates, names)]
    rewards = []
    choices = []
    alpha = np.ones(len(bandits))  # Beta prior: α=1, β=1 (uniform)
    beta  = np.ones(len(bandits))
    for _ in range(n_rounds):
        # Sample from each arm's posterior
        samples = [np.random.beta(alpha[i], beta[i]) for i in range(len(bandits))]
        arm = np.argmax(samples)
        reward = bandits[arm].pull()
        alpha[arm] += reward
        beta[arm]  += (1 - reward)
        rewards.append(reward); choices.append(arm)
    return rewards, choices, bandits

# Run all strategies
np.random.seed(42)
rew_eg, cho_eg, bands_eg = epsilon_greedy(None, n_rounds, epsilon=0.1)
rew_ucb, cho_ucb, bands_ucb = ucb1(None, n_rounds)
rew_ts, cho_ts, bands_ts = thompson_sampling(None, n_rounds)

# Optimal reward (if we always picked best arm)
optimal_total = max(true_rates) * n_rounds

print('=== Multi-Armed Bandit Results ===')
print(f'n_rounds: {n_rounds}')
print(f'Optimal (always pick best): {optimal_total:.0f} conversions')
print()
for name, rewards, choices in [
    ('Epsilon-Greedy (ε=0.1)', rew_eg, cho_eg),
    ('UCB1', rew_ucb, cho_ucb),
    ('Thompson Sampling', rew_ts, cho_ts),
]:
    total = sum(rewards)
    regret = optimal_total - total
    best_pct = choices.count(3) / n_rounds * 100  # arm D is index 3
    print(f'{name:<30} conversions={total:.0f}  regret={regret:.0f}  best_arm%={best_pct:.1f}%')

# Cumulative regret plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
best_rate = max(true_rates)

for label, rewards, color in [
    ('ε-Greedy', rew_eg, 'orange'),
    ('UCB1', rew_ucb, 'steelblue'),
    ('Thompson', rew_ts, 'green'),
]:
    cumulative_regret = np.cumsum([best_rate - r for r in rewards])
    axes[0].plot(cumulative_regret, label=label, color=color, lw=2)

axes[0].set_xlabel('Round'); axes[0].set_ylabel('Cumulative Regret')
axes[0].set_title('Cumulative Regret (lower=better)'); axes[0].legend()

# Arm selection distribution
strategies = {'ε-Greedy': cho_eg, 'UCB1': cho_ucb, 'Thompson': cho_ts}
x_pos = np.arange(len(names))
width = 0.25
for i, (label, choices) in enumerate(strategies.items()):
    counts = [choices.count(j)/n_rounds*100 for j in range(4)]
    axes[1].bar(x_pos + i*width, counts, width, label=label, alpha=0.8)
axes[1].set_xticks(x_pos + width)
axes[1].set_xticklabels(names, rotation=15)
axes[1].set_ylabel('% of pulls'); axes[1].set_title('Arm Selection Distribution')
axes[1].legend()

plt.tight_layout(); plt.show()

## 5. Sample Size Calculation & Power Analysis

In [ ]:
from scipy.stats import norm

def sample_size_proportion(p1, p2, alpha=0.05, power=0.80, two_tailed=True):
    """Calculate required sample size per variant for proportion A/B test."""
    p_avg = (p1 + p2) / 2
    z_alpha = norm.ppf(1 - alpha / (2 if two_tailed else 1))
    z_beta  = norm.ppf(power)
    
    n = ((z_alpha * np.sqrt(2 * p_avg * (1 - p_avg)) +
           z_beta  * np.sqrt(p1 * (1-p1) + p2 * (1-p2)))**2 / (p1 - p2)**2)
    return int(np.ceil(n))

# Sample size vs MDE (minimum detectable effect)
p_base = 0.10
lifts = np.linspace(0.05, 0.50, 50)  # relative lifts from 5% to 50%
p_treat_vals = p_base * (1 + lifts)

n_80  = [sample_size_proportion(p_base, pt, power=0.80) for pt in p_treat_vals]
n_90  = [sample_size_proportion(p_base, pt, power=0.90) for pt in p_treat_vals]
n_95  = [sample_size_proportion(p_base, pt, power=0.95) for pt in p_treat_vals]

plt.figure(figsize=(10, 5))
plt.plot(lifts*100, n_80, 'b-', lw=2, label='Power=80%')
plt.plot(lifts*100, n_90, 'r--', lw=2, label='Power=90%')
plt.plot(lifts*100, n_95, 'g:', lw=2, label='Power=95%')
plt.xlabel('Relative Lift (%)')
plt.ylabel('Sample Size per Variant')
plt.title(f'Required Sample Size vs Effect Size (baseline={p_base:.0%}, α=0.05)')
plt.yscale('log')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

print('Sample size examples (baseline=10%, α=0.05, power=80%):')
for lift in [0.05, 0.10, 0.20, 0.50]:
    pt = p_base * (1 + lift)
    n  = sample_size_proportion(p_base, pt)
    print(f'  {lift:+.0%} lift ({p_base:.0%}→{pt:.2%}): {n:,} per variant  ({n*2:,} total)')

print()
print('Key A/B testing pitfalls:')
print('  1. PEEKING — stop when p<0.05 → inflated FPR (use SPRT or Bayesian)')
print('  2. MULTIPLE COMPARISONS — testing many metrics → use Bonferroni/FDR')
print('  3. NOVELTY EFFECT — users behave differently because things changed')
print('  4. SEGMENT IMBALANCE — SRM (Sample Ratio Mismatch) check always')
print('  5. NETWORK EFFECTS — Uber/Airbnb: variants interact through network')